In [ ]:
import os, sys
import socket
from pathlib import Path
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt

from vi_rnn.saving import load_model
from vi_rnn.inference import filtering_posterior_bootstrap
from vi_rnn.datasets import SWM_dataset_multi
from vi_rnn.generate import generate
from evaluation.eval_spikestats import calc_stats
from fig_utils.plots import plot_spike_histogram_stats
from fig_utils.spike_stats import (
    eval_spike_stats,
    gather_spike_histogram_stats,
)

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

# change to your own path

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn_cl/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# --- controls ---

# --- run ---
min_data_points_isi = 5
eval_train = True
run = True  # also computes held-out bootstrap-SMC log-likelihood

In [ ]:
if run:

    rows = []

    for model_dir in model_dirs:
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=True, backward_compat=False
        )
        task_params["path"] = path

        ll_final = float(training_params["ll"][-1])
        n_steps = int(len(training_params["ll"]))
        task = SWM_dataset_multi(task_params)
        spike_stats = eval_spike_stats(
            vae,
            task,
            min_data_points_isi=min_data_points_isi,
            eval_train=eval_train,
            k=training_params.get("k", 32),
            encoder_padding=training_params.get("encoder_padding", 0),
        )

        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "ll_final": ll_final,
                "n_steps": n_steps,
                "macaque": task_params["sessions"][0][5:10],
                "pwcorr": spike_stats["r2_pwcorr"],
                "mean_rate": spike_stats["mean_rate"],
                "std_ISI": spike_stats["std_ISI"],
                "ll": spike_stats["ll"],
            }
        )
        if eval_train:
            rows[-1]["mean_rate_train"] = spike_stats["mean_rate_train"]
            rows[-1]["std_ISI_train"] = spike_stats["std_ISI_train"]
            rows[-1]["pwcorr_train"] = spike_stats["r2_pwcorr_train"]
            rows[-1]["ll_train"] = spike_stats["ll_train"]

In [ ]:
if run:
    df = pd.DataFrame(rows)

    df = df.assign(
        pwcorr_mean=df["pwcorr"].apply(
            lambda x: np.mean(x) if isinstance(x, list) else x
        )
    ).sort_values(["macaque", "pwcorr_mean"], ascending=[True, True]).reset_index(
        drop=True
    )

    df.to_pickle("../data/processed/df_spike_stats_data.pkl")
else:
    df = pd.read_pickle("../data/processed/df_spike_stats_data.pkl")

In [ ]:
%matplotlib inline

In [ ]:
pwc_corr_train_m1 = []
pwc_corr_test_m1 = []
rate_train_m1 = []
rate_test_m1 = []
ISI_train_m1 = []
ISI_test_m1 = []
ll_train_m1 = []
ll_test_m1 = []

pwc_corr_train_m2 = []
pwc_corr_test_m2 = []
rate_train_m2 = []
rate_test_m2 = []
ISI_train_m2 = []
ISI_test_m2 = []
ll_train_m2 = []
ll_test_m2 = []

mq1 = "groot"
mq2 = "ocean"

mc1_inds = np.where(df["macaque"] == mq1)[0]
mc2_inds = np.where(df["macaque"] == mq2)[0]

for i in range(15):
    pwc_corr_train_m1.append(np.mean([df["pwcorr_train"][m][i] for m in mc1_inds]))
    pwc_corr_test_m1.append(np.mean([df["pwcorr"][m][i] for m in mc1_inds]))
    rate_train_m1.append(np.mean([df["mean_rate_train"][m][i] for m in mc1_inds]))
    rate_test_m1.append(np.mean([df["mean_rate"][m][i] for m in mc1_inds]))
    ISI_train_m1.append(np.mean([df["std_ISI_train"][m][i] for m in mc1_inds]))
    ISI_test_m1.append(np.mean([df["std_ISI"][m][i] for m in mc1_inds]))
    ll_train_m1.append(np.mean([df["ll_train"][m][i] for m in mc1_inds]))
    ll_test_m1.append(np.mean([df["ll"][m][i] for m in mc1_inds]))

    pwc_corr_train_m2.append(np.mean([df["pwcorr_train"][m][i] for m in mc2_inds]))
    pwc_corr_test_m2.append(np.mean([df["pwcorr"][m][i] for m in mc2_inds]))
    rate_train_m2.append(np.mean([df["mean_rate_train"][m][i] for m in mc2_inds]))
    rate_test_m2.append(np.mean([df["mean_rate"][m][i] for m in mc2_inds]))
    ISI_train_m2.append(np.mean([df["std_ISI_train"][m][i] for m in mc2_inds]))
    ISI_test_m2.append(np.mean([df["std_ISI"][m][i] for m in mc2_inds]))
    ll_train_m2.append(np.mean([df["ll_train"][m][i] for m in mc2_inds]))
    ll_test_m2.append(np.mean([df["ll"][m][i] for m in mc2_inds]))


fig, ax = plt.subplots(2, 3, sharex=True, sharey=True)

ax[0, 0].plot(pwc_corr_train_m1, label="train")
ax[0, 0].plot(pwc_corr_test_m1, label="test")
ax[0, 1].plot(rate_train_m1, label="train")
ax[0, 1].plot(rate_test_m1, label="test")
ax[0, 2].plot(ISI_train_m1, label="train")
ax[0, 2].plot(ISI_test_m1, label="test")

ax[1, 0].plot(pwc_corr_train_m2, label="train")
ax[1, 0].plot(pwc_corr_test_m2, label="test")
ax[1, 1].plot(rate_train_m2, label="train")
ax[1, 1].plot(rate_test_m2, label="test")
ax[1, 2].plot(ISI_train_m2, label="train")
ax[1, 2].plot(ISI_test_m2, label="test")

ax[0, 0].set_title("pw_corr")
ax[0, 1].set_title("mean rate (Hz)")
ax[0, 2].set_title("std ISI (s)")
ax[0, 0].set_ylabel("mac " + str(mq1))
ax[1, 0].set_ylabel("mac " + str(mq2))
ax[0, 0].legend()
ax[1, 0].set_xlabel("session")

plt.show()

# need formatting max 3 decimals:
print(
    f"mu+sd pwc corr groot {np.mean(pwc_corr_test_m1):.3f} {np.std(pwc_corr_test_m1):.3f}"
)
print(
    f"mu+sd pw corr ocean {np.mean(pwc_corr_test_m2):.3f} {np.std(pwc_corr_test_m2):.3f}"
)

print(f"mu+sd mean rate groot {np.mean(rate_test_m1):.3f} {np.std(rate_test_m1):.3f}")
print(f"mu+sd mean rate ocean {np.mean(rate_test_m2):.3f} {np.std(rate_test_m2):.3f}")

print(f"mu+sd std ISI groot {np.mean(ISI_test_m1):.3f} {np.std(ISI_test_m1):.3f}")
print(f"mu+sd std ISI ocean {np.mean(ISI_test_m2):.3f} {np.std(ISI_test_m2):.3f}")

fig, ax = plt.subplots(1, 2, sharex=True, sharey=False)
ax[0].plot(ll_train_m1, label="train")
ax[0].plot(ll_test_m1, label="test")
ax[1].plot(ll_train_m2, label="train")
ax[1].plot(ll_test_m2, label="test")
ax[0].set_title("LL, " + mq1)
ax[1].set_title("LL, " + mq2)
ax[0].set_ylabel("LL (nats/bin)")
ax[0].set_xlabel("session")
ax[1].set_xlabel("session")
ax[0].legend()
plt.tight_layout()
plt.show()

print(
    f"mu+sd test LL groot {np.mean(ll_test_m1):.3f} {np.std(ll_test_m1):.3f}"
)
print(
    f"mu+sd test LL ocean {np.mean(ll_test_m2):.3f} {np.std(ll_test_m2):.3f}"
)


## Spike histograms (one model, all sessions)

Same panels as synthetic supplement: ISI CV per unit, mean ISI per unit, pairwise correlation histogram. Uses a subset of validation trials per session.

In [ ]:
# pick one trained model (same layout as loop above)
model_dir = out_dir / model_dirs[0]
vae, training_params, task_params = load_model(
    str(model_dir), load_encoder=True, backward_compat=False
)
task_params["path"] = path
task = SWM_dataset_multi(task_params)
print(model_dir.name, "n_sessions:", len(task.sessions))

In [ ]:
max_trials_per_session = 250
session_list = list(range(len(task.sessions)))

# recorded spikes (data)
stats_data = gather_spike_histogram_stats(
    task,
    use_generated=False,
    max_trials_per_session=max_trials_per_session,
    session_list=session_list,
)
plot_spike_histogram_stats(
    stats_data,
    save_path="../paper_figures/swm_spike_stats_data.png",
)

In [ ]:
# model-generated spikes
stats_gen = gather_spike_histogram_stats(
    task,
    vae,
    use_generated=True,
    max_trials_per_session=max_trials_per_session,
    session_list=session_list,
)
plot_spike_histogram_stats(
    stats_gen,
    save_path="../paper_figures/swm_spike_stats_generated.png",
)